In [1]:
%pip install -q langgraph langchain-google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import operator
import math
import datetime
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage,
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY not found. Add it to your .env file.")

chat_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    google_api_key=gemini_api_key,
)

import langgraph as _lg
from importlib.metadata import version
print("\u2705  Setup complete!")
print(f"   LangGraph : {version('langgraph')}")
print(f"   Model     : gemini-2.5-flash")

# Quick sanity check
_test = chat_model.invoke([HumanMessage(content="Say hello in one word.")])
print(f"   Test call : {_test.content.strip()}")

In [ ]:
# ── Basic StateGraph: nodes, fixed edges, START / END ────────────────────
print("PART A - LangGraph: StateGraph Basics")
print("=" * 55)
# ── 1. State schema ───────────────────────────────────────────────────────
class PipelineState(TypedDict):
    text: str          # provided at invocation time
    char_count: int    # populated by node 1
    summary: str       # populated by node 2

# ── 2. Node functions ─────────────────────────────────────────────────────
def count_chars(state: PipelineState) -> dict:
    """Node 1: count characters in the input text."""
    return {"char_count": len(state["text"])}

def make_summary(state: PipelineState) -> dict:
    """Node 2: build a human-readable summary string."""
    s = f"'{state['text']}' \u2014 {state['char_count']} characters."
    return {"summary": s}


# ── 3. Build the graph ────────────────────────────────────────────────────
builder_basic = StateGraph(PipelineState)
builder_basic.add_node("count",     count_chars)
builder_basic.add_node("summarise", make_summary)

# ── 4. Wire edges: START \u2192 count \u2192 summarise \u2192 END ─────────────────────────────
builder_basic.add_edge(START,      "count")
builder_basic.add_edge("count",    "summarise")
builder_basic.add_edge("summarise", END)

# ── 5. Compile and run ────────────────────────────────────────────────────
graph_basic = builder_basic.compile()
result = graph_basic.invoke({
    "text": "LangGraph makes stateful AI workflows easy!",
    "char_count": 0,
    "summary": "",
})

print("Input      :", result["text"])
print("Char count :", result["char_count"])
print("Summary    :", result["summary"])
print()
print("Node execution order: START \u2192 count \u2192 summarise \u2192 END")

In [ ]:
# ── LLM-Powered Pipeline: Summarise \u2192 Translate ─────────────────────────────
print("\u2500\u2500 LLM Pipeline: Summarise \u2192 Translate to French \u2500\u2500\n")

class NLPState(TypedDict):
    article:     str
    summary:     str
    translation: str
def summarise_node(state: NLPState) -> dict:
    """Node 1: Summarise the article to 2 sentences."""
    prompt = f"Summarise the following text in exactly 2 sentences:\n\n{state['article']}"
    resp = chat_model.invoke([HumanMessage(content=prompt)])
    return {"summary": resp.content.strip()}

def translate_node(state: NLPState) -> dict:
    """Node 2: Translate the summary to French."""
    prompt = f"Translate this text to French (keep it natural):\n\n{state['summary']}"
    resp = chat_model.invoke([HumanMessage(content=prompt)])
    return {"translation": resp.content.strip()}

builder_nlp = StateGraph(NLPState)
builder_nlp.add_node("summarise", summarise_node)
builder_nlp.add_node("translate", translate_node)

builder_nlp.add_edge(START,       "summarise")
builder_nlp.add_edge("summarise", "translate")
builder_nlp.add_edge("translate",  END)

nlp_graph = builder_nlp.compile()

article = (
    "Artificial intelligence is transforming industries worldwide. "
    "From healthcare diagnostics to autonomous vehicles, AI systems are "
    "performing tasks that previously required human expertise. "
    "The rapid adoption of large language models has further accelerated this trend."
)
r = nlp_graph.invoke({"article": article, "summary": "", "translation": ""})
print("Original Article:\n", article, "\n")
print("Summary:\n", r["summary"], "\n")
print("Translation (French):\n", r["translation"], "\n")

In [ ]:
# ── Conditional Routing: Math vs General Knowledge ──────────────────────
print("\u2500\u2500 Conditional Routing: Math vs General Knowledge \u2500\u2500\n")

class RouterState(TypedDict):
    question: str
    category: str    # set by classify_node
    answer:   str    # set by the specialist node

MATH_KEYWORDS = {
    "calculate", "compute", "plus", "minus", "multiply", "times",
    "divide", "square root", "percent", "sum", "how much is", 
}

def classify_node(state: RouterState) -> dict:
    """Classify the question as math or general knowledge."""
    q = state["question"].lower()
    cat = "math" if any(kw in q for kw in MATH_KEYWORDS) else "general"
    return {"category": cat}


def math_node(state: RouterState) -> dict:
    resp = chat_model.invoke([
        SystemMessage(content="You are a concise mathematics assistant."),
        HumanMessage(content=state["question"]),
    ])
    return {"answer": resp.content.strip()}

def route_question(state: RouterState) -> Literal["math_node", "general_node"]:
    """Routing function: return the next node name based on category."""
    return f"{state['category']}_node"

def general_node(state: RouterState) -> dict:
    resp = chat_model.invoke([
        SystemMessage(content="You are a concise general-knowledge assistant."),
        HumanMessage(content=state["question"]),
    ])
    return {"answer": resp.content.strip()}

builder_router = StateGraph(RouterState)
builder_router.add_node("classify",     classify_node)
builder_router.add_node("math_node",    math_node)
builder_router.add_node("general_node", general_node)

builder_router.add_edge(START, "classify")
builder_router.add_conditional_edges("classify", route_question)   # \u2190 routing
builder_router.add_edge("math_node",    END)
builder_router.add_edge("general_node", END)

router_graph = builder_router.compile()
questions = [
    "What is the square root of 144?",
    "Who wrote the novel 1984?",
    "Calculate 256 divided by 8",
    "What is the capital of Australia?",
]
for q in questions:
    r = router_graph.invoke({"question": q, "category": "", "answer": ""})
    print(f"\u2753 {q}")
    print(f"   Category : {r['category']}")
    print(f"   Answer   : {r['answer'][:150].strip()}")
    print()